# USC Keck: clinical data analysis

This notebook shows the filtering, reshaping, plotting, date calculations, and nearest-neighbor methods I worked with during my USC Keck research.

All records below are synthetic. Their dates, stages, and measurements are made up to demonstrate the code. The plots should not be used to draw clinical conclusions.

The ViT image example and document converter are in separate files.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


## Create the example data

The notebook creates the records here, so there is no file to upload.


In [ ]:
# Entirely invented rows for a public software demonstration.
rows = []
for i in range(1, 7):
    person = f"demo-{i:03d}"
    def add_event(tag, fields, suffix=""):
        for field, value in fields.items():
            rows.append({"id": person, "tag": tag,
                         "tag_id": f"{person}-{tag}{suffix}",
                         "field": field, "value": value})
    surgery = pd.Timestamp("2024-01-01") + pd.Timedelta(days=i * 10)
    progression = surgery + pd.Timedelta(days=120 + i * 40)
    add_event("surgery", {"date": surgery.strftime("%Y-%m-%d")})
    add_event("biochemical-progression", {"date": progression.strftime("%Y-%m-%d")})
    add_event("gleason-grade-group", {"value": (i % 5) + 1})
    add_event("psa-t", {"value": 1.5 + i})
    add_event("pT", {"T": ["2a", "2b", "2c", "3a", "3b", "3c"][i - 1]})
    add_event("pN", {"N": str(i % 2)})
    add_event("M", {"M": ["0", "1a", "1b", "1c", "0", "1a"][i - 1]})
    for visit in range(3):
        add_event("psa", {"date": (surgery + pd.Timedelta(days=30 * visit)).strftime("%Y-%m-%d"),
                          "value": round(0.2 * i + 0.05 * visit, 2), "units": "ng/ml"},
                  suffix=f"-{visit}")
df = pd.DataFrame(rows)
df.head()


## Plot stage counts


In [ ]:
import matplotlib.pyplot as plt

def histogram(field_name, tag_name):
    # Filter the DataFrame based on the given criteria
    filtered_df = df[(df['field'] == field_name) & (df['tag'] == tag_name)]
    print("Filtered DataFrame:\n", filtered_df)
    # Predefined stages
    if (field_name=='T' and tag_name=='pT'):
        stages = ['2a', '2b', '2c', '3a', '3b', '3c']
    elif (field_name=='N' and tag_name=='pN'):
        stages=['0', '1']
    elif (field_name=='M' and tag_name=='M'):
        stages=['0', '1a', '1b', '1c']
    else:
        raise ValueError('Unsupported stage field/tag pair')
    # Generate the stage distribution with predefined stages
    stage_distribution = {stage: 0 for stage in stages}
    stage_counts = filtered_df['value'].value_counts().to_dict()

    for stage, count in stage_counts.items():
        if stage in stage_distribution:
            stage_distribution[stage] = count

    # Plot the histogram
    plt.bar(stage_distribution.keys(), stage_distribution.values())
    plt.xlabel('Stage')
    plt.ylabel('Count')
    plt.title(f'Distribution of Stages for field "{field_name}" and tag "{tag_name}"')
    plt.show()

histogram('M', 'M')


## Filter and reshape records


In [ ]:
#find specific rows given 1 specific criteria for each group
def find_rows(df, filter_dict):
    # Initialize the mask to all True
    mask = pd.Series(True, index=df.index)

    # Iterate over the filter dictionary to build the mask
    for column, value in filter_dict.items():
        mask = mask & (df[column] == value)

    # Return the filtered DataFrame
    return df[mask]

#find rows where 'tag' is 'psa' and 'field' is 'field'
filter_dict = {"tag": "gleason-grade-group", "field" : "value"}
result = find_rows(df, filter_dict)




#finds the info of a patient and reshapes it such that the values under field column becomes a column
def find_values(df, patient_id):
    filtered_df = df[df['id'] == patient_id]
    # print(filtered_df)
    reshaped_df = filtered_df.pivot(index=['id', 'tag_id'], columns='field', values='value').reset_index()
    return reshaped_df


# find_values(result, "demo-003")


result


In [ ]:
filter = {
    'tag': ['surgery', 'biochemical-progression'],
    'field': 'date',
}

#find patient info returns the date of biochemical-progression and surgery
def find_patientinfo(df, filter_dict):
    # df_patientid = df[df['id'] == patient_id]

    # Create a filter for tags
    tag_filter = df['tag'].isin(filter_dict['tag'])

    # Apply the filter for 'field_name'
    field_filter = df['field'] == filter_dict['field']

    # Apply both filters
    df_filtered = df[tag_filter & field_filter]

    # Reshape the DataFrame
    reshaped_df = df_filtered.pivot(index=['id'], columns='tag', values='value').reset_index()

    return reshaped_df


## Calculate time intervals and plot the data

`days_till_failure` is the original variable name for the difference between the two event dates. The calculation does not account for censoring.


In [ ]:
patientinfo=find_patientinfo(df, filter)
patientinfo['surgery'] = pd.to_datetime(patientinfo['surgery'])
patientinfo['biochemical-progression'] = pd.to_datetime(patientinfo['biochemical-progression'])
resultdays = (patientinfo['biochemical-progression'] - patientinfo['surgery']).dt.days
patientinfo['days_till_failure']=resultdays
#Merging to create correlation matrix
merged_df = pd.merge(result, patientinfo, on='id')
correlation_data = merged_df[['value', 'days_till_failure']]
correlation_data
# correlation_matrix = correlation_data.corr()
# print(correlation_matrix)


In [ ]:
#Histogram plotting relationship between gleason grade group and days till failure
filterpsa={"tag":"psa"}
resultpsa=find_rows(df, filterpsa)
res_psa_date_and_value=find_values(resultpsa, "demo-003")
res_psa_date_and_value['value'] = pd.to_numeric(res_psa_date_and_value['value'])
res_psa_date_and_value['date'] = pd.to_datetime(res_psa_date_and_value['date'])
first_appointment_date = res_psa_date_and_value['date'].min()
res_psa_date_and_value['days_from_first_appointment'] = (res_psa_date_and_value['date'] - first_appointment_date).dt.days
res_psa_date_and_value
plt.plot(res_psa_date_and_value['days_from_first_appointment'], res_psa_date_and_value['value'])
plt.xlabel('Days from First Appointment')
plt.ylabel('Value')
plt.title('Line Plot of Days from First Appointment vs. Value')
plt.grid(True)
plt.show()


In [ ]:
#Creating Box and Whisker Plots to demonstrate the relationship between categorical values[{tag:”pT”, “field”:”T”} {tag:”pN”, “field”:”N”} {tag:”M”, “field”:”M”} and numerical values target (time between bio-chmemical progression and surgery)
filtercategorical1={"tag":"pT", "field":"T"}
filtercategorical2={"tag":"pN", "field":"N"}
filtercategorical3={"tag":"M", "field":"M"}
resultcategorical1=find_rows(df, filtercategorical1)
resultcategorical2=find_rows(df, filtercategorical2)
resultcategorical3=find_rows(df, filtercategorical3)
patientinfoforplots=find_patientinfo(df, filter)
patientinfoforplots['surgery'] = pd.to_datetime(patientinfoforplots['surgery'])
patientinfoforplots['biochemical-progression'] = pd.to_datetime(patientinfoforplots['biochemical-progression'])
resultdays2 = (patientinfoforplots['biochemical-progression'] - patientinfoforplots['surgery']).dt.days
patientinfoforplots['days_till_failure']=resultdays2
merged_df1 = pd.merge(resultcategorical1, patientinfoforplots, on='id')
merged_df2 = pd.merge(resultcategorical2, patientinfoforplots, on='id')
merged_df3 = pd.merge(resultcategorical3, patientinfoforplots, on='id')
# Convert 'value' column to categorical type
merged_df1['value'] = merged_df1['value'].astype('category')
merged_df2['value'] = merged_df2['value'].astype('category')
merged_df3['value'] = merged_df3['value'].astype('category')
# Create the box-and-whisker plot
plt.figure(figsize=(20, 6))
plt.subplot(1, 3, 1)
sns.boxplot(x='value', y='days_till_failure', data=merged_df1)
plt.xlabel('Value for tag pT and field T')
plt.ylabel('Days till Failure')
plt.title('Box-and-Whisker Plot of Days till Failure by Value')
merged_df2
plt.subplot(1, 3, 2)
sns.boxplot(x='value', y='days_till_failure', data=merged_df2)
plt.xlabel('Value for tag pN and field N')
plt.ylabel('Days till Failure')
plt.title('Box-and-Whisker Plot of Days till Failure by Value')
plt.subplot(1, 3, 3)
sns.boxplot(x='value', y='days_till_failure', data=merged_df3)
plt.xlabel('Value for tag M and field M')
plt.ylabel('Days till Failure')
plt.title('Box-and-Whisker Plot of Days till Failure by Value')
plt.show()


## Find the nearest record

This example scales two variables and finds the closest record by distance. It is a simple demonstration rather than a validated clinical matching model.


In [ ]:
filtermodel = {
    'tag': ['psa-t', 'gleason-grade-group'],
    'field': 'value',
}
dataset=find_patientinfo(df, filtermodel)
dataset
ids = dataset['id'].values
dataset['gleason-grade-group'] = pd.to_numeric(dataset['gleason-grade-group'], errors='raise')
dataset['psa-t'] = pd.to_numeric(dataset['psa-t'], errors='coerce')
psa_mean = dataset['psa-t'].mean()
dataset['psa-t'] = dataset['psa-t'].fillna(psa_mean)

features_to_scale = ['gleason-grade-group', 'psa-t']
scaler = StandardScaler()
data_scaled = scaler.fit_transform(dataset[features_to_scale])
new_patient = pd.DataFrame({
    'gleason-grade-group': [4],  # Example input for Gleason Grade Group
    'psa-t': [6.0]  # Example input for PSA-T value
})
new_patient_scaled = scaler.transform(new_patient[features_to_scale])
knn = NearestNeighbors(n_neighbors=1)
knn.fit(data_scaled)
distance, index = knn.kneighbors(new_patient_scaled)
nearest_id = ids[index[0][0]]
print(f"The most similar patient ID is: {nearest_id}")
